In [1]:
import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
import joblib
import pandas as pd
import imageio
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from selenium import webdriver
from time import sleep
import os
import re 
from IPython.display import display
from glob import glob as gb
import streamlit as st
from PIL import Image
import io
import base64

C:\Users\Debra\Anaconda3\envs\geo\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [ ]:
input_folder = r'C:\Users\GIF file'  # Folder containing Kyle19841231.jpg, Kyle19851231.jpg, etc.
output_gif = 'kyle_dam_changes.gif'

#Years for analysis
years_to_iterate = [1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000,
                    2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017,
                    2018, 2019, 2020]
#years_to_iterate = [2020]

# Directory repository for classified images
directory = r'C:\Users\classification images'

In [3]:
def check_and_read_image(year, directory = directory):
    filename = f'classified_{year}.tif'
    filepath = os.path.join(directory, filename)
    if os.path.exists(filepath):
        return imageio.imread(filepath)
    else:
        print(f'Warning: file {filename} does not exist.')
        return None
    
def gif_creation(out_gif, years_to_iterate, duration = 1):
    images = [check_and_read_image(year) for year in years_to_iterate]
    images = [img for img in images if img is not None]
    
    if images:
        # Create GIF if images are available
        imageio.mimsave(out_gif, images, duration)
        print(f'GIF saved to {out_gif}')
    else:
        print('No images available to create GIF')

### Creating gif for the dam level changes and visualisation.

In [4]:
# Configuration
duration_per_frame = int(60000 / 37)  # 60,000ms (1min) divided by 37 frames ≈ 1622ms per frame
font_path = 'arial.ttf'  # Path to font file or use default
font_size = 24

# 1. Load and sort the images
image_files = sorted(gb(os.path.join(input_folder, 'Kyle*.jpg')))
images = []

for img_path in image_files:
    # Extract year from filename (assuming format KyleYYYY1231.jpg)
    year = os.path.basename(img_path)[4:8]
    
    # Open image and add year annotation
    img = Image.open(img_path)
    draw = ImageDraw.Draw(img)
    
    try:
        font = ImageFont.truetype(font_path, font_size)
    except:
        font = ImageFont.load_default()
    
    # Add year text (white with black outline for visibility)
    text = f"Year: {year}"
    text_position = (10, 10)
    
    # Draw outline
    outline_color = "black"
    for x_offset in [-1, 0, 1]:
        for y_offset in [-1, 0, 1]:
            draw.text((text_position[0] + x_offset, text_position[1] + y_offset), 
                      text, font=font, fill=outline_color)
    
    # Draw main text
    draw.text(text_position, text, font=font, fill="white")
    
    images.append(img)

# 2. Save as GIF
images[0].save(
    output_gif,
    save_all=True,
    append_images=images[1:],
    duration=duration_per_frame,
    loop=0  # Infinite loop
)

print(f"GIF created successfully: {output_gif}")

GIF created successfully: kyle_dam_changes.gif


In [5]:
# import geemap
import matplotlib.pyplot as plt
import numpy as np
# import joblib
import pandas as pd
# import imageio
from pathlib import Path

In [ ]:
# 1. Load the Excel file

file_path = r"C:\Users\mutirikwi dataset\MUTIRIKWI DAM.xlsx"

df = pd.read_excel(file_path)

print("File loaded successfully!")
print(f"Shape: {df.shape}\n")



File loaded successfully!
Shape: (3243, 4)



In [7]:
# 2. Check for null values
print("=== Null Values ===")
null_counts = df.isnull().sum()
print(null_counts)
print("\n")



=== Null Values ===
DAM_NAME     0
WEEKDATE     0
DAM_LEVEL    0
DAM_PRCAP    0
dtype: int64




In [11]:
#pip install --upgrade pandas
import pandas as pd
print(pd.__version__)


3.0.2


In [12]:
print("=== Summary Statistics ===")
print(df.describe(include='all'))

=== Summary Statistics ===
         DAM_NAME                    WEEKDATE    DAM_LEVEL     DAM_PRCAP
count        3243                        3243  3243.000000   3243.000000
unique          2                         NaN          NaN           NaN
top     MUTIRIKWI                         NaN          NaN           NaN
freq         3242                         NaN          NaN           NaN
mean          NaN  1999-10-19 16:03:59.777983    91.812547    791.057586
min           NaN         1970-01-05 00:00:00    10.970000      0.000000
25%           NaN         1988-03-21 12:00:00    86.040000    375.958000
50%           NaN         2003-06-16 00:00:00    93.080000    749.553000
75%           NaN         2011-07-02 12:00:00    99.670000   1253.720000
max           NaN         2019-05-17 00:00:00   103.190000  14190.000000
std           NaN                         NaN     8.548073    510.492054


In [13]:
# Check datetime columns separately
datetime_cols = df.select_dtypes(include=['datetime64[ns]'])
print(datetime_cols.describe())

                         WEEKDATE
count                        3243
mean   1999-10-19 16:03:59.777983
min           1970-01-05 00:00:00
25%           1988-03-21 12:00:00
50%           2003-06-16 00:00:00
75%           2011-07-02 12:00:00
max           2019-05-17 00:00:00


In [14]:
# 4. Check for duplicates
print("=== Duplicates ===")
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")





=== Duplicates ===
Number of duplicate rows: 0


In [15]:
# 5. Save processed data
output_file = "processed_data.xlsx"
df.to_excel(output_file, index=False)
print(f"Processed data saved to {output_file}")

Processed data saved to processed_data.xlsx
